# Transparent Decomposition and Verification
Prerequisites: integer arithmetic and assertions. Estimated time: 30 minutes. Objectives: decompose a small arithmetic task into auditable steps, verify the answer independently, and detect a deliberately injected error.

## Concept recap and assumptions
A prompt may ask a model for intermediate reasoning, but model-generated explanations are not guaranteed to expose its internal process. This notebook uses an explicit, human-authored arithmetic decomposition as a teaching aid and checks its result with an independent calculation; it makes no claim about hidden chain-of-thought access.

In [ ]:
def solve_total(items):
    # Return a concise, inspectable ledger and final total.
    subtotal = items['notebooks'] * items['notebook_price']
    shipping = items['shipping']
    total = subtotal + shipping
    explanation = [f"notebooks: {items['notebooks']} x {items['notebook_price']} = {subtotal}", f"shipping: {shipping}", f"total: {subtotal} + {shipping} = {total}"]
    return total, explanation

order = {'notebooks': 3, 'notebook_price': 12, 'shipping': 5}
answer, steps = solve_total(order)
steps

## Baseline assertions
The decomposition is useful because each arithmetic statement can be checked. The verifier recomputes from the original inputs, rather than trusting the displayed steps.

In [ ]:
def verify_total(items, proposed):
    return proposed == items['notebooks'] * items['notebook_price'] + items['shipping']

assert answer == 41
assert verify_total(order, answer)
assert len(steps) == 3

## Comparison and metric
For this synthetic arithmetic task, exact correctness is the metric. A displayed explanation that is plausible but inconsistent with the inputs must fail verification. In an LLM application, evaluate final-answer correctness and explanation faithfulness separately; do not infer that a fluent rationale reveals private internal reasoning.

In [ ]:
# Failure cases: arithmetic slip, missing field, and invalid negative quantity.
assert not verify_total(order, 40)  # injected addition error
try:
    solve_total({'notebooks': 1, 'notebook_price': 10})
except KeyError:
    missing_input_rejected = True
else:
    missing_input_rejected = False
assert missing_input_rejected
invalid_order = {'notebooks': -1, 'notebook_price': 10, 'shipping': 2}
assert verify_total(invalid_order, -8)  # arithmetic can be correct even when input violates policy
assert invalid_order['notebooks'] < 0  # domain validation is a separate check
assert order['notebooks'] >= 0  # valid business-rule check

## Exercises
1. Add tax as a separate line and update the verifier independently. 2. Reject non-integer prices and negative quantities before solving. 3. Create a wrong intermediate subtotal with the right final answer and explain why checking only the final number is insufficient. Expected properties: verification uses original inputs, and domain checks are separate from arithmetic checks.

## Summary, next lesson, and practice
Transparent decomposition can make a toy solution auditable, while an independent verifier catches errors. This does not expose or validate a model's hidden reasoning. Next: RAG and evaluation. Practice: [Chain-of-Thought interview drill](../interview-prep/curriculum-practice.md#chain-of-thought-cot).